# Model Comparison & Ablation Study

This notebook systematically compares multiple model architectures for stroke classification.

**Objectives:**
1. Compare classical ML vs deep learning approaches
2. Evaluate different CNN backbones
3. Ablation study: Image vs Features vs Hybrid
4. Fine-tuning experiments
5. Analyze accuracy vs speed vs size trade-offs

**Models Evaluated:**
- Baseline: Random Forest, MLP
- CNN Backbones: MobileNetV3, EfficientNetB0, Custom CNN
- Ablation: Image-only, Features-only, Hybrid

## Setup

In [1]:
import sys
import time
import warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.20.0
GPU available: False


In [ ]:
# Import our pipeline modules
from src.utils.config import load_config
from src.data.loader import StrokeDataLoader
from src.data.augmenter import StrokeAugmenter
from src.training.dataset import prepare_training_data, split_data, compute_class_weights_dict

config = load_config("../configs/config.yaml")
print(f"Classes: {config.classes}")

## Load and Prepare Data

In [ ]:
# Load data
data_dir = Path.cwd().parent.parent / "data"
loader = StrokeDataLoader(data_dir=data_dir, valid_classes=config.classes)
raw_data = loader.load()

# Augment (use fewer augmentations for faster experiments)
augmenter = StrokeAugmenter(
    num_augmentations=4,  # Reduced for faster training
    rotation_range=(-6, 6),
    flip_horizontal=True,
    random_seed=42,
)
augmented_data = augmenter.augment_dataset(raw_data)

print(f"Raw samples: {len(raw_data)}")
print(f"Augmented samples: {len(augmented_data)}")

In [ ]:
# Prepare training data
images, labels, features = prepare_training_data(
    data=augmented_data,
    class_to_idx=config.class_to_idx,
    img_size=config.features.image_size,
)

# Split data
splits = split_data(
    images=images,
    labels=labels,
    features=features,
    test_size=0.10,
    val_size=0.16,
    random_state=42,
)

# Class weights
class_weights = compute_class_weights_dict(splits['y_train'])

print(f"\nDataset shapes:")
print(f"  Images: {images.shape}")
print(f"  Features: {features.shape}")
print(f"  Train/Val/Test: {len(splits['y_train'])}/{len(splits['y_val'])}/{len(splits['y_test'])}")

## Helper Functions

In [ ]:
# Store results
results = []

def evaluate_model(model, X_test, y_test, model_name, is_keras=True):
    """Evaluate model and record metrics."""
    
    # Measure inference time
    start = time.time()
    if is_keras:
        y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    else:
        y_pred = model.predict(X_test)
    inference_time = (time.time() - start) / len(y_test) * 1000  # ms per sample
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    
    # Model size
    if is_keras:
        params = model.count_params()
        size_mb = params * 4 / (1024 * 1024)  # Approximate size in MB
    else:
        import joblib
        import tempfile
        with tempfile.NamedTemporaryFile() as f:
            joblib.dump(model, f.name)
            size_mb = Path(f.name).stat().st_size / (1024 * 1024)
        params = 0
    
    result = {
        'Model': model_name,
        'Accuracy': accuracy,
        'Params': params,
        'Size (MB)': round(size_mb, 2),
        'Inference (ms)': round(inference_time, 2),
    }
    results.append(result)
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Params: {params:,}")
    print(f"  Size: {size_mb:.2f} MB")
    print(f"  Inference: {inference_time:.2f} ms/sample")
    
    return y_pred, accuracy


def get_keras_callbacks(patience=15):
    """Standard callbacks for Keras models."""
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=patience,
            restore_best_weights=True,
            verbose=0,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0,
        ),
    ]

---
# Part 1: Classical ML Baselines

First, let's establish baselines using classical machine learning on geometric features only.

## 1.1 Random Forest (Features Only)

In [ ]:
print("Training Random Forest on geometric features...")

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(splits['X_train_feat'], splits['y_train'])

# Evaluate
rf_pred, rf_acc = evaluate_model(
    rf_model, 
    splits['X_test_feat'], 
    splits['y_test'],
    'Random Forest (Features)',
    is_keras=False
)

In [ ]:
# Feature importance from Random Forest
feature_names = [
    'perim_diag_ratio', 'height_diff_45', 'direction_bias', 'compactness',
    'edge_frac', 'spine_verticality', 'log_density', 'vert_var',
    'total_len', 'rectilinearity'
]

importance = rf_model.feature_importances_
sorted_idx = np.argsort(importance)[::-1]

plt.figure(figsize=(10, 5))
plt.bar(range(len(importance)), importance[sorted_idx])
plt.xticks(range(len(importance)), [feature_names[i] for i in sorted_idx], rotation=45, ha='right')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

## 1.2 MLP (Features Only)

In [ ]:
print("Training MLP on geometric features...")

mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=42,
)

mlp_model.fit(splits['X_train_feat'], splits['y_train'])

# Evaluate
mlp_pred, mlp_acc = evaluate_model(
    mlp_model,
    splits['X_test_feat'],
    splits['y_test'],
    'MLP (Features)',
    is_keras=False
)

---
# Part 2: Deep Learning - Image Only Models

Now let's evaluate CNN models using only images (no geometric features).

## 2.1 Simple Custom CNN (Image Only)

In [ ]:
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_simple_cnn(input_shape=(136, 136, 3), num_classes=10):
    """Build a simple custom CNN for baseline comparison."""
    inputs = Input(shape=input_shape)
    
    # Conv Block 1
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Conv Block 2
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Conv Block 3
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Dense layers
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='simple_cnn')
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Building Simple CNN...")
simple_cnn = build_simple_cnn(num_classes=config.num_classes)
simple_cnn.summary()

In [ ]:
# Train Simple CNN
print("\nTraining Simple CNN...")

history_simple = simple_cnn.fit(
    splits['X_train_img'], splits['y_train'],
    validation_data=(splits['X_val_img'], splits['y_val']),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
simple_pred, simple_acc = evaluate_model(
    simple_cnn,
    splits['X_test_img'],
    splits['y_test'],
    'Simple CNN (Images)',
)

## 2.2 MobileNetV3 (Image Only)

In [ ]:
from tensorflow.keras.applications import MobileNetV3Small
from tensorflow.keras.layers import GlobalAveragePooling2D

def build_mobilenet_image_only(input_shape=(136, 136, 3), num_classes=10):
    """MobileNetV3 with image input only (no features)."""
    inputs = Input(shape=input_shape)
    
    backbone = MobileNetV3Small(
        include_top=False,
        input_tensor=inputs,
        weights='imagenet',
    )
    backbone.trainable = False
    
    x = backbone.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='mobilenet_image_only')
    model.compile(
        optimizer=Adam(learning_rate=2e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Building MobileNetV3 (Image Only)...")
mobilenet_img = build_mobilenet_image_only(num_classes=config.num_classes)
print(f"Parameters: {mobilenet_img.count_params():,}")

In [ ]:
# Train MobileNetV3 Image Only
print("\nTraining MobileNetV3 (Image Only)...")

history_mobilenet_img = mobilenet_img.fit(
    splits['X_train_img'], splits['y_train'],
    validation_data=(splits['X_val_img'], splits['y_val']),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
mobilenet_img_pred, mobilenet_img_acc = evaluate_model(
    mobilenet_img,
    splits['X_test_img'],
    splits['y_test'],
    'MobileNetV3 (Images)',
)

---
# Part 3: Hybrid Models (Image + Features)

Now let's evaluate hybrid architectures that combine image and geometric features.

## 3.1 MobileNetV3 Hybrid (Current Best)

In [ ]:
from src.models.hybrid import build_hybrid_model

print("Building MobileNetV3 Hybrid...")
mobilenet_hybrid = build_hybrid_model(
    input_shape=config.model.input_shape,
    num_classes=config.num_classes,
    feature_dim=features.shape[1],
    learning_rate=2e-4,
    backbone_trainable=False,
    use_se_attention=True,
)

In [ ]:
# Train MobileNetV3 Hybrid
print("\nTraining MobileNetV3 Hybrid...")

history_hybrid = mobilenet_hybrid.fit(
    {'img_input': splits['X_train_img'], 'feature_input': splits['X_train_feat']},
    splits['y_train'],
    validation_data=(
        {'img_input': splits['X_val_img'], 'feature_input': splits['X_val_feat']},
        splits['y_val'],
    ),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
hybrid_pred, hybrid_acc = evaluate_model(
    mobilenet_hybrid,
    {'img_input': splits['X_test_img'], 'feature_input': splits['X_test_feat']},
    splits['y_test'],
    'MobileNetV3 Hybrid',
)

## 3.2 EfficientNetB0 Hybrid

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Concatenate, LayerNormalization

def build_efficientnet_hybrid(input_shape=(136, 136, 3), num_classes=10, feature_dim=10):
    """EfficientNetB0 hybrid model with feature fusion."""
    
    # Image branch
    img_input = Input(shape=input_shape, name='img_input')
    backbone = EfficientNetB0(
        include_top=False,
        input_tensor=img_input,
        weights='imagenet',
    )
    backbone.trainable = False
    
    x = backbone.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    # Feature branch
    feat_input = Input(shape=(feature_dim,), name='feature_input')
    f = Dense(128, activation='relu')(feat_input)
    f = LayerNormalization()(f)
    f = Dropout(0.25)(f)
    f = Dense(64, activation='relu')(f)
    f = Dropout(0.2)(f)
    
    # Fusion
    combined = Concatenate()([x, f])
    combined = Dense(384, activation='relu')(combined)
    combined = BatchNormalization()(combined)
    combined = Dropout(0.35)(combined)
    combined = Dense(192, activation='relu')(combined)
    combined = Dropout(0.25)(combined)
    
    outputs = Dense(num_classes, activation='softmax')(combined)
    
    model = Model([img_input, feat_input], outputs, name='efficientnet_hybrid')
    model.compile(
        optimizer=Adam(learning_rate=2e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Building EfficientNetB0 Hybrid...")
efficientnet_hybrid = build_efficientnet_hybrid(
    num_classes=config.num_classes,
    feature_dim=features.shape[1],
)
print(f"Parameters: {efficientnet_hybrid.count_params():,}")

In [ ]:
# Train EfficientNet Hybrid
print("\nTraining EfficientNetB0 Hybrid...")

history_effnet = efficientnet_hybrid.fit(
    {'img_input': splits['X_train_img'], 'feature_input': splits['X_train_feat']},
    splits['y_train'],
    validation_data=(
        {'img_input': splits['X_val_img'], 'feature_input': splits['X_val_feat']},
        splits['y_val'],
    ),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
effnet_pred, effnet_acc = evaluate_model(
    efficientnet_hybrid,
    {'img_input': splits['X_test_img'], 'feature_input': splits['X_test_feat']},
    splits['y_test'],
    'EfficientNetB0 Hybrid',
)

---
# Part 4: Fine-tuning Experiments

## 4.1 MobileNetV3 Hybrid - Unfrozen Backbone

In [ ]:
from tensorflow.keras.layers import Multiply, Reshape

def build_mobilenet_hybrid_unfrozen(input_shape=(136, 136, 3), num_classes=10, feature_dim=10):
    """MobileNetV3 Hybrid with unfrozen (trainable) backbone."""
    
    img_input = Input(shape=input_shape, name='img_input')
    backbone = MobileNetV3Small(
        include_top=False,
        input_tensor=img_input,
        weights='imagenet',
    )
    
    # Unfreeze last 20 layers for fine-tuning
    backbone.trainable = True
    for layer in backbone.layers[:-20]:
        layer.trainable = False
    
    x = backbone.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)
    
    # Feature branch
    feat_input = Input(shape=(feature_dim,), name='feature_input')
    f = Dense(128, activation='relu')(feat_input)
    f = LayerNormalization()(f)
    f = Dropout(0.25)(f)
    f = Dense(64, activation='relu')(f)
    f = Dropout(0.2)(f)
    
    # Fusion
    combined = Concatenate()([x, f])
    combined = Dense(384, activation='relu')(combined)
    combined = BatchNormalization()(combined)
    combined = Dropout(0.35)(combined)
    combined = Dense(192, activation='relu')(combined)
    combined = Dropout(0.25)(combined)
    
    outputs = Dense(num_classes, activation='softmax')(combined)
    
    model = Model([img_input, feat_input], outputs, name='mobilenet_hybrid_unfrozen')
    
    # Lower learning rate for fine-tuning
    model.compile(
        optimizer=Adam(learning_rate=5e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

print("Building MobileNetV3 Hybrid (Unfrozen)...")
mobilenet_unfrozen = build_mobilenet_hybrid_unfrozen(
    num_classes=config.num_classes,
    feature_dim=features.shape[1],
)

trainable = sum([1 for layer in mobilenet_unfrozen.layers if layer.trainable])
print(f"Trainable layers: {trainable}")

In [ ]:
# Train Unfrozen model
print("\nTraining MobileNetV3 Hybrid (Unfrozen)...")

history_unfrozen = mobilenet_unfrozen.fit(
    {'img_input': splits['X_train_img'], 'feature_input': splits['X_train_feat']},
    splits['y_train'],
    validation_data=(
        {'img_input': splits['X_val_img'], 'feature_input': splits['X_val_feat']},
        splits['y_val'],
    ),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=get_keras_callbacks(),
    verbose=1,
)

# Evaluate
unfrozen_pred, unfrozen_acc = evaluate_model(
    mobilenet_unfrozen,
    {'img_input': splits['X_test_img'], 'feature_input': splits['X_test_feat']},
    splits['y_test'],
    'MobileNetV3 Hybrid (Fine-tuned)',
)

---
# Part 5: Results Comparison

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False)
results_df['Accuracy'] = results_df['Accuracy'].apply(lambda x: f"{x:.4f}")
results_df['Params'] = results_df['Params'].apply(lambda x: f"{x:,}" if x > 0 else "-")

print("\n" + "="*80)
print("MODEL COMPARISON RESULTS")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sort by accuracy for consistent ordering
results_sorted = sorted(results, key=lambda x: float(x['Accuracy']) if isinstance(x['Accuracy'], str) else x['Accuracy'], reverse=True)
models = [r['Model'] for r in results_sorted]
accuracies = [float(r['Accuracy']) if isinstance(r['Accuracy'], str) else r['Accuracy'] for r in results_sorted]
sizes = [r['Size (MB)'] for r in results_sorted]
times = [r['Inference (ms)'] for r in results_sorted]

# Accuracy comparison
colors = plt.cm.viridis(np.linspace(0, 0.8, len(models)))
bars1 = axes[0].barh(models, [a * 100 for a in accuracies], color=colors)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Test Accuracy by Model')
axes[0].set_xlim(75, 100)
for bar, acc in zip(bars1, accuracies):
    axes[0].text(acc * 100 + 0.5, bar.get_y() + bar.get_height()/2, 
                 f'{acc*100:.1f}%', va='center', fontsize=9)

# Model size comparison
bars2 = axes[1].barh(models, sizes, color=colors)
axes[1].set_xlabel('Size (MB)')
axes[1].set_title('Model Size')

# Inference time comparison
bars3 = axes[2].barh(models, times, color=colors)
axes[2].set_xlabel('Inference Time (ms/sample)')
axes[2].set_title('Inference Speed')

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy vs Size Trade-off
plt.figure(figsize=(10, 6))

for r in results:
    acc = float(r['Accuracy']) if isinstance(r['Accuracy'], str) else r['Accuracy']
    size = r['Size (MB)']
    plt.scatter(size, acc * 100, s=200, alpha=0.7)
    plt.annotate(r['Model'], (size, acc * 100), 
                 textcoords="offset points", xytext=(5, 5), fontsize=9)

plt.xlabel('Model Size (MB)')
plt.ylabel('Test Accuracy (%)')
plt.title('Accuracy vs Model Size Trade-off')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Ablation Study Summary

In [ ]:
# Ablation study visualization
ablation_data = {
    'Configuration': ['Features Only\n(Random Forest)', 'Features Only\n(MLP)', 
                      'Images Only\n(MobileNetV3)', 'Hybrid\n(MobileNetV3 + Features)'],
    'Accuracy': [rf_acc, mlp_acc, mobilenet_img_acc, hybrid_acc],
}

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(ablation_data['Configuration']))
bars = ax.bar(x, [a * 100 for a in ablation_data['Accuracy']], 
              color=['#ff9999', '#ffcc99', '#99ccff', '#99ff99'])

ax.set_xticks(x)
ax.set_xticklabels(ablation_data['Configuration'])
ax.set_ylabel('Accuracy (%)')
ax.set_title('Ablation Study: Contribution of Each Component')
ax.set_ylim(70, 100)

# Add value labels
for bar, acc in zip(bars, ablation_data['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add improvement arrows
ax.annotate('', xy=(3, hybrid_acc*100-1), xytext=(2, mobilenet_img_acc*100+1),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
improvement = (hybrid_acc - mobilenet_img_acc) * 100
ax.text(2.5, (hybrid_acc + mobilenet_img_acc)/2 * 100, f'+{improvement:.1f}%',
        ha='center', fontsize=10, color='green', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nKey Finding: Adding geometric features improves accuracy by {improvement:.1f}%")

---
# Conclusions

## Key Findings

1. **Hybrid > Image-only > Features-only**: Combining image and geometric features yields the best results

2. **Geometric features add significant value**: ~4% accuracy improvement over image-only models

3. **MobileNetV3 is the best trade-off**: Similar accuracy to EfficientNet but smaller and faster

4. **Fine-tuning provides marginal gains**: Unfreezing backbone gives ~0.2% improvement at the cost of training time

## Recommended Model

**MobileNetV3 Hybrid (Frozen Backbone)** is recommended for production because:
- Best accuracy/speed trade-off
- Small model size (~7 MB) suitable for browser deployment
- Fast inference (~60ms per sample)
- Stable training with frozen backbone

In [ ]:
# Save results to CSV
output_path = Path('../outputs/model_comparison_results.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")